<a href="https://colab.research.google.com/github/mscids2024pranita-hue/goa-rainfall-arima/blob/main/05_sarima_model_fitting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 — SARIMA Model Fitting
**Project:** Goa Rainfall Forecasting using SARIMA
**Model:** SARIMA(0,0,0)(1,1,1,12)
**Step:** Train the model on 80% of data, check model summary

In [1]:
!pip install netCDF4 xarray statsmodels -q

import zipfile, os, re
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX

from google.colab import files
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall("imd_data")

DATA_DIR = "imd_data/imd_rainfall-20260513T063252Z-3-001/imd_rainfall/"
GOA_LAT_MIN, GOA_LAT_MAX = 14.9, 15.7
GOA_LON_MIN, GOA_LON_MAX = 73.7, 74.3

files_nc = sorted(os.listdir(DATA_DIR))
files_nc = [f for f in files_nc if re.match(r'RF25_ind\d{4}_rfp25\.nc$', f)]

daily_list = []
for fname in files_nc:
    ds_yr = xr.open_dataset(DATA_DIR + fname)
    goa_yr = ds_yr['RAINFALL'].sel(LATITUDE=slice(GOA_LAT_MIN, GOA_LAT_MAX), LONGITUDE=slice(GOA_LON_MIN, GOA_LON_MAX))
    goa_daily_yr = goa_yr.mean(dim=['LATITUDE','LONGITUDE'], skipna=True)
    times = pd.DatetimeIndex(ds_yr['TIME'].values)
    daily_list.append(pd.Series(goa_daily_yr.values, index=times))
    ds_yr.close()

daily_goa = pd.concat(daily_list).sort_index()
monthly_goa = daily_goa.resample('MS').sum()

for month in range(1, 13):
    monthly_goa[f'2005-{month:02d}-01'] = (monthly_goa[f'2004-{month:02d}-01'] + monthly_goa[f'2006-{month:02d}-01']) / 2

monthly_goa.name = 'rainfall_mm'
df = monthly_goa.to_frame()
print(f"Ready. Shape: {df.shape}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 26.2 MB/s eta 0:00:00


Saving imd_rainfall-20260513T063252Z-3-001-20260530T092135Z-3-001.zip to imd_rainfall-20260513T063252Z-3-001-20260530T092135Z-3-001.zip
Ready. Shape: (420, 1)


In [2]:
# 80% train, 20% test
train_size = int(len(df) * 0.8)
train = df['rainfall_mm'][:train_size]
test  = df['rainfall_mm'][train_size:]

print(f"Total months : {len(df)}")
print(f"Train months : {len(train)}  ({train.index[0].strftime('%b %Y')} to {train.index[-1].strftime('%b %Y')})")
print(f"Test months  : {len(test)}   ({test.index[0].strftime('%b %Y')} to {test.index[-1].strftime('%b %Y')})")

Total months : 420
Train months : 336  (Jan 1991 to Dec 2018)
Test months  : 84   (Jan 2019 to Dec 2025)


In [3]:
# Fit SARIMA(0,0,0)(1,1,1,12)
model = SARIMAX(train,
                order=(0, 0, 0),
                seasonal_order=(1, 1, 1, 12),
                enforce_stationarity=False,
                enforce_invertibility=False)

fitted_model = model.fit(disp=False)

print(fitted_model.summary())

                                 SARIMAX Results                                  
Dep. Variable:                rainfall_mm   No. Observations:                  336
Model:             SARIMAX(1, 1, [1], 12)   Log Likelihood               -2073.638
Date:                    Tue, 02 Jun 2026   AIC                           4153.275
Time:                            16:51:00   BIC                           4164.495
Sample:                        01-01-1991   HQIC                          4157.760
                             - 12-01-2018                                         
Covariance Type:                      opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.S.L12       0.0837      0.037      2.264      0.024       0.011       0.156
ma.S.L12      -0.7953      0.026    -30.295      0.000      -0.847      -0.744
sigma2      3.556e+0

In [4]:
with open('07_sarima_model_summary.txt', 'w') as f:
    f.write(str(fitted_model.summary()))

print("Model summary saved.")
print(f"\nAIC : {fitted_model.aic:.3f}")
print(f"BIC : {fitted_model.bic:.3f}")

Model summary saved.

AIC : 4153.275
BIC : 4164.495
